In [ ]:
import os
with open("./job.list", "w") as f:
    for file in sorted(os.listdir("./inputs")):
        if file.endswith(".pdb"):
            filename = file[:-4]
        f.write( "/home/chuwang_pkuhpc/lustre1/jobs/cjj/install/rosetta.source.release-425/main/source/bin/relax.mpi.linuxgccrelease "
                 f"-s ./inputs/{file} "
                 "-relax:constrain_relax_to_start_coords "
                 "-ramp_constraints false "
                 "-relax:coord_constrain_sidechains "
                 "-nstruct 40 "
                 "-out:path:all ./outputs "
                 "-out:file:silent_struct_type binary "
                f"-out:file:silent {filename}.out.gz "
                f"-out:file:scorefile {filename}.sc "
                 "-ex1 "
                 "-ex2 "
                 "-ignore_zero_occupancy false "
                 "-use_input_sc "
                 "-flip_HNQ "
                 "-no_optH false "
                 "-score:weights beta_jan25 "
                f"-beta_jan25 > ./outputs/logs/{filename}_relaxed.log 2>&1\n" )

In [2]:
import os
with open("interface.list", "w") as f:
    for file in sorted(os.listdir("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/Dock/4_GLM/validation/generalization/rosetta/outputs")):
        if file.endswith(".pdb"):
            path = os.path.join("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/Dock/4_GLM/validation/generalization/rosetta/outputs", file)
            f.write(f"InterfaceAnalyzer.mpi.linuxgccrelease -s {path} -out:file:score_only /home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/Dock/4_GLM/validation/generalization/rosetta/interface/packed_interface_score-betajan25.sc -beta_jan25 @/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/Dock/4_GLM/validation/interface/pack_input_options_nopackstat.txt > /home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/Dock/4_GLM/validation/generalization/rosetta/interface/{file}.log 2>&1\n")


In [3]:
import os

# 检查input中的每个文件是否都有40个输出结果
input_files = sorted([f for f in os.listdir("./inputs") if f.endswith(".pdb")])
output_files = sorted([f for f in os.listdir("./outputs/sc") if f.endswith(".sc")])
lost_num = 0
dub_num = 0
lost_files = []
dub_files = []
for input_file in input_files:
    base_name = input_file[:-4]  # 去掉 .pdb后缀
    matching_outputs = f'{base_name}.sc'
    if not os.path.exists(f"./outputs/sc/{matching_outputs}"):
        # print(f"文件 {input_file} 的输出结果缺失: 没有找到 {matching_outputs}")
        lost_num += 1
        lost_files.append(input_file)
    else:
        with open(f"./outputs/sc/{matching_outputs}", "r") as f:
            lines = f.readlines()
            if len(lines) > 80:
                # print(f"文件 {input_file} 的输出结果数量不正确: 找到 {len(lines)} 行，预期 42 行")
                dub_num += 1
                dub_files.append(input_file)

print(f"总共有 {lost_num} 个输入文件的输出结果缺失。")
print(f"总共有 {dub_num} 个输入文件重复操作了。")


总共有 452 个输入文件的输出结果缺失。
总共有 518 个输入文件重复操作了。


In [3]:
with open("remove_file.sh", "w") as f:
    for file in dub_files:
        base_name = file[:-4]
        f.write(f"rm {base_name}*\n")

In [5]:
# 检查/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface-classifier/3_AF3/relax/outputs/logs中的CMD行的pdb对应的file名称是否与dub_files完全一致
log_files = sorted([f for f in os.listdir("./outputs/logs") if f.endswith(".std")])
count = 0
for log_file in log_files:
    with open(f"./outputs/logs/{log_file}", "r") as f:
        lines = f.readlines()
        for line in lines:
            if line.startswith("CMD:"):
                cmd_parts = line.split()
                for part in cmd_parts:
                    if part.endswith(".pdb"):
                        pdb_file = os.path.basename(part)
                        if pdb_file in dub_files:
                            # print(f"文件 {pdb_file} 在日志 {log_file} 中被重复操作了。")
                            count += 1
print(f"总共有 {count} 个输入文件在日志中被重复操作了。")

总共有 518 个输入文件在日志中被重复操作了。


In [6]:
# 将没有完成的内容写入新的job_rest.list文件
with open("./job_1-518.list", "w") as f:
    for file in dub_files:
        if file.endswith(".pdb"):
            filename = file[:-4]
        f.write( "/home/chuwang_pkuhpc/lustre1/jobs/cjj/install/rosetta.source.release-425/main/source/bin/relax.mpi.linuxgccrelease "
                 f"-s ./inputs/{file} "
                 "-relax:constrain_relax_to_start_coords "
                 "-ramp_constraints false "
                 "-relax:coord_constrain_sidechains "
                 "-nstruct 40 "
                 "-out:path:all ./outputs "
                 "-out:file:silent_struct_type binary "
                f"-out:file:silent {filename}.out.gz "
                f"-out:file:scorefile {filename}.sc "
                 "-ex1 "
                 "-ex2 "
                 "-ignore_zero_occupancy false "
                 "-use_input_sc "
                 "-flip_HNQ "
                 "-no_optH false "
                 "-score:weights beta_jan25 "
                f"-beta_jan25 > ./outputs/logs/{filename}_relaxed.log 2>&1\n" )

In [4]:
# 提取每个sc文件中total_score最低的description，将对应的description组成一个list
import os
import pandas as pd 
pdb = set(["_".join(pdb.split("_")[:-1]) for pdb in os.listdir("./outputs/file") if pdb.endswith(".pdb")])
descriptions = []
done_files = ["_".join(d.split("_")[:-1]) + ".sc" for d in os.listdir("./outputs/file") if d.endswith(".pdb")]
for file in sorted(os.listdir("./outputs/sc")):
    # if file in done_files:
    #     continue
    # if file.endswith(".sc") and (not file.startswith("6g5g")) and (not file.startswith("5e33")) and (not file.startswith("6e5x")):
    df = pd.read_csv(f"./outputs/sc/{file}", skiprows=1, sep=r'\s+')
    min_score_row = df.loc[df['total_score'].idxmin()]
    descriptions.append(min_score_row['description'])
print(len(descriptions))


2226


In [5]:
with open("./outputs/file/extract_pdb_rest.sh", "w") as f:
    for description in descriptions:
        file = "_".join(description.split("_")[:-1])
        # f.write(f"/home/chuwang_pkuhpc/lustre1/jobs/cjj/install/rosetta.source.release-425/main/source/bin/extract_pdbs.mpi.linuxgccrelease -in:file:silent ./outputs/{file}.out.gz -in:file:tags \"{description}\"\n")
        f.write(f"extract_pdbs.mpi.linuxgccrelease -in:file:silent ../silent/{file}.out.gz -in:file:tags \"{description}\"\n")

In [7]:
with open("./interface/jobs.list", "w") as f:
    for description in descriptions:
        f.write(f"InterfaceAnalyzer.mpi.linuxgccrelease -s /home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface-classifier/AF3/relax/outputs/file/{description}.pdb -out:file:score_only af3_samples-interface-betajan25-nopackinput.sc -beta_jan25 @/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface-classifier/Dock/flags/pack_separate_options.txt > /home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface-classifier/AF3/relax/interface/logs/{description}_pack_separated.log 2>&1\n")

In [ ]:
import os
sc = set([sc.split(".")[0] for sc in os.listdir("./outputs/sc") if sc.endswith(".sc")])
pdb = set(["_".join(pdb.split("_")[:-1]) for pdb in os.listdir("./outputs/file") if pdb.endswith(".pdb")])

diff = sc - pdb
print(len(diff))
# print(list(diff)[1])



        

1422
1a0n_53_seed-45_sample-1_model
